# Preparation of data.

In [1]:
import os
import json
import pandas as pd
from tqdm import tqdm
from pathlib import Path

In [5]:
RAW_DATA_DIR = Path("data/01_raw_c")
PROCESSED_DATA_DIR = Path("data/02_processed")
OUTPUT_CSV_FILE = PROCESSED_DATA_DIR / "clean_cve_data.csv"

In [6]:
def safe_get(data, path):
    """Safely navigates a nested structure of dictionaries and lists."""
    current_level = data
    for key in path:
        try:
            if isinstance(current_level, list):
                current_level = current_level[key]
            elif isinstance(current_level, dict):
                current_level = current_level.get(key)
            else:
                return None
        except (IndexError, KeyError, TypeError):
            return None
    return current_level

In [7]:
def prepare_clean_data():
    print("Starting Data Preparation Process...")
    PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
    
    if not RAW_DATA_DIR.exists():
        print(f"ERROR: Raw data directory not found at {RAW_DATA_DIR}")
        print("Please run script 01_data_acquisition.ipynb first.")
        return

    all_records = []
    
    for year_folder in sorted(os.listdir(RAW_DATA_DIR)):
        year_path = RAW_DATA_DIR / year_folder
        if not year_path.is_dir():
            continue

        files_to_process = [f for f in year_path.glob("*.json")]
        
        for file_path in tqdm(files_to_process, desc=f"Parsing {year_folder}"):
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                    
                    description_path = ["cve", "descriptions", 0, "value"]
                    severity_path_v31 = ["cve", "metrics", "cvssMetricV31", 0, "cvssData", "baseSeverity"]
                    score_path_v31 = ["cve", "metrics", "cvssMetricV31", 0, "cvssData", "baseScore"]
                    cwe_path = ["cve", "weaknesses", 0, "description", 0, "value"]

                    record = {
                        "CVE_ID": safe_get(data, ["cve", "id"]),
                        "Description": safe_get(data, description_path),
                        "Severity": safe_get(data, severity_path_v31),
                        "CVSS_Score": safe_get(data, score_path_v31),
                        "CWE": safe_get(data, cwe_path),
                        "Year": year_folder
                    }
                    all_records.append(record)
                    
                except json.JSONDecodeError:
                    print(f"Warning: Could not parse {file_path}, skipping.")

    df = pd.DataFrame(all_records)

    print("\nCleaning and inspecting the data...")
    
    df.dropna(subset=["CVE_ID", "Description", "Severity"], inplace=True)
    
    initial_rows = len(df)
    df.drop_duplicates(subset=["Description"], keep='first', inplace=True)
    final_rows = len(df)
    print(f"Removed {initial_rows - final_rows} duplicate entries based on description.")

    df["CWE"].fillna("N/A", inplace=True)

    print("\nDataframe Info:")
    df.info()

    print("\nClass Distribution (Target Variable 'Severity'):")
    print(df["Severity"].value_counts())
    
    df.to_csv(OUTPUT_CSV_FILE, index=False, encoding="utf-8")
    print(f"\nClean data preparation complete!")
    print(f"Processed data saved to: {OUTPUT_CSV_FILE}")

if __name__ == "__main__":
    prepare_clean_data()

Starting Data Preparation Process...


Parsing 2025: 100%|██████████| 1269/1269 [00:10<00:00, 123.22it/s]



Cleaning and inspecting the data...
Removed 295 duplicate entries based on description.

Dataframe Info:
<class 'pandas.core.frame.DataFrame'>
Index: 7212 entries, 5 to 12978
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CVE_ID       7212 non-null   object 
 1   Description  7212 non-null   object 
 2   Severity     7212 non-null   object 
 3   CVSS_Score   7212 non-null   float64
 4   CWE          7212 non-null   object 
 5   Year         7212 non-null   object 
dtypes: float64(1), object(5)
memory usage: 394.4+ KB

Class Distribution (Target Variable 'Severity'):
Severity
MEDIUM      4031
HIGH        2297
CRITICAL     495
LOW          389
Name: count, dtype: int64

Clean data preparation complete!
Processed data saved to: data\02_processed\clean_cve_data.csv


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21256\952382235.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["CWE"].fillna("N/A", inplace=True)
